# Notebook 04 — Baselines

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 04 of 12

---

### Why this notebook decides whether the research has a result

Section 5.6 of the self-check guidelines requires comparison against *the strongest reasonable
alternative, not the easiest one to beat*. For this project that bar is unusually specific, because
notebook 03 established that **capture statistics alone reach macro F1 0.295 on the four-class task
and 0.565 on binary detection** under the adopted augmentation policy, against majority baselines
of 0.164 and 0.328.

That is the real floor. A convolutional network scoring 0.60 on binary detection would have
demonstrated nothing at all — a gradient-boosted tree reaches that from image statistics with no
pixels. **Every number in Chapter 5 must be read against this floor rather than against chance**,
and that reframing is itself a contribution, because the egg-inspection literature reports raw
accuracy against chance and would not catch this.

Five baselines are built, spanning from trivially weak to not-deployable-but-informative.

| | Baseline | What it tells us |
|---|---|---|
| B0 | Majority class and stratified random | The arithmetic floor |
| B1 | Capture statistics only | **The honest floor.** Anything at or below this is reading the camera, not the egg |
| B2 | Classical texture descriptors (LBP, Gabor) + linear classifier | What generic texture analysis achieves without any crack-specific modelling |
| B3 | Automatic ridge-based crack segmentation, then geometry | A purpose-built classical crack pipeline. The strongest non-deep alternative, and the one a reviewer will ask about |
| B4 | Human annotation polygon geometry | **Not deployable.** An oracle ceiling: how well the morphology taxonomy can be resolved when the crack is already perfectly located |

B3 and B4 together answer the question the whole project rests on. Notebook 01 showed that human
polygons separate the three morphologies at macro F1 0.789 using geometry alone. If an automatic
ridge detector can recover enough of that geometry, a classical pipeline is a genuine competitor
and the deep model must beat it. If it cannot, the gap between B3 and B4 quantifies exactly how
much of the problem is *finding* the crack rather than *classifying* it — which is the argument for
a learned approach, and it directs where notebook 05 should invest.

Every baseline uses **the identical fold assignment** written by notebook 02, so the numbers are
directly comparable to the models in notebook 05 without any caveat. The sealed test set is not
touched.

---
## 0. Configuration

In [ ]:
import os
os.environ["EGG_ARTEFACT_ROOT"] = r"E:\Degree\Final Research\Egg_Crack_Dataset\Eggshell-Diagnostic-Research\Artefact"

CACHE_VARIANT = "neutral_fill"
CACHE_SIDE    = 448
FEAT_SIDE     = 256    # features are extracted at 256 px; ridge filtering at 384 is slow and the
                       # classical descriptors gain nothing from the extra resolution
N_JOBS        = -1
RIDGE_SIGMAS  = (1.0, 2.0, 3.0)      # scales at which a crack behaves as a bright ridge
DARK_SIGMAS   = (3.0, 5.0, 7.0)      # larger scales: the egg silhouette, which must be suppressed
SILHOUETTE_PCTL = 97.0               # dark-ridge percentile treated as silhouette
BORDER_FRAC   = 0.04                 # border band zeroed before thresholding
MIN_COMPONENT_PX = 15                # smallest accepted crack fragment, in feature-resolution px
RIDGE_PCTL    = 98.0                 # default; overridden by the sweep in section 2

In [ ]:
import sys, json, math, warnings, time
from pathlib import Path
from datetime import datetime

_here = Path.cwd()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _src is None:
    raise RuntimeError("Run from Artefact/notebooks/ or set EGG_ARTEFACT_ROOT.")
sys.path.insert(0, str(_src))

import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from skimage.feature import local_binary_pattern
from skimage.filters import sato
from skimage.morphology import remove_small_objects

from common import project_config as CFG
from common.augmentation import standardise

warnings.filterwarnings("ignore")
CFG.seed_everything()

STAGE = "04_baselines"
OUT = CFG.results_dir(STAGE); FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PROC = CFG.DATA_DIR / "processed"

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})
_N = {"i": 0}; FIGS, TABS = [], []
def savefig(fig, name, cap=""):
    _N["i"] += 1; f = f"{_N['i']:02d}_{name}.png"
    fig.savefig(FIG_DIR / f, bbox_inches="tight"); FIGS.append({"file": f, "caption": cap})
    print(f"  [figure] {f}")
def savetable(df, name, cap=""):
    p = TAB_DIR / f"{name}.csv"; df.to_csv(p); TABS.append({"file": p.name, "caption": cap})
    print(f"  [table ] {p.name}  {df.shape}"); return df
S = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds")}

splits = pd.read_csv(PROC / "splits.csv")
cache = np.load(PROC / f"cache_{CACHE_VARIANT}_{CACHE_SIDE}.npy", mmap_mode="r")
dev = splits[splits["split"].eq("dev")].reset_index(drop=True)
dev["label4"] = pd.Categorical(dev["label4"], categories=CFG.LABEL4, ordered=True)

y4 = dev["label4"].astype(str).to_numpy()
y2 = dev["is_cracked"].to_numpy()
folds = dev["fold"].to_numpy()
blocks = dev["capture_block"].to_numpy()
print(f"development set {len(dev)} eggs, folds {sorted(set(folds))}")
print(f"class counts {dev['label4'].value_counts().to_dict()}")
print("the sealed test set is NOT touched in this notebook")

---
## 1. Feature extraction

Three descriptor families, all computed on the per-image standardised images so that the
photometric leak neutralised in notebook 02 is not quietly reintroduced through the features.

**Local Binary Patterns** capture micro-texture: the mottled shell surface against the sharper
linear discontinuity of a crack. Two radii, uniform patterns, normalised histograms.

**Gabor energy** responds to oriented linear structure at several scales, which is exactly what a
hairline crack is. Four orientations and three wavelengths; mean and standard deviation of each
response magnitude.

**Ridge filtering** (Sato tubeness) is the purpose-built one. Under transillumination a crack
appears as a *bright* line against the shell, so the filter is run with `black_ridges=False` across
four scales. This both supplies features and produces the automatic segmentation used by B3.

In [ ]:
def prep_gray(i):
    """Standardised grayscale at the feature resolution, requantised to uint8."""
    im = cv2.resize(np.asarray(cache[i]), (FEAT_SIDE, FEAT_SIDE), interpolation=cv2.INTER_AREA)
    f = standardise(im).mean(axis=2)
    lo, hi = np.percentile(f, [0.5, 99.5])
    return np.clip((f - lo) / max(hi - lo, 1e-6) * 255, 0, 255).astype(np.uint8)


GABOR_BANK = [cv2.getGaborKernel((21, 21), sigma, theta, lam, 0.5, 0, ktype=cv2.CV_32F)
              for theta in np.arange(0, np.pi, np.pi / 4)
              for sigma, lam in [(2.0, 6.0), (3.0, 10.0), (4.5, 16.0)]]


def texture_features(g):
    out = {}
    # --- local binary patterns
    for P, R in [(8, 1), (16, 2)]:
        lbp = local_binary_pattern(g, P, R, method="uniform")
        hist, _ = np.histogram(lbp, bins=P + 2, range=(0, P + 2), density=True)
        for k, v in enumerate(hist):
            out[f"lbp{P}_{k}"] = float(v)
    # --- gabor energy
    gf = g.astype(np.float32)
    for k, kern in enumerate(GABOR_BANK):
        r = cv2.filter2D(gf, cv2.CV_32F, kern)
        out[f"gabor{k}_mean"] = float(np.abs(r).mean())
        out[f"gabor{k}_std"] = float(r.std())
    return out


def ridge_response(g):
    """Bright-ridge (crack) response with the egg silhouette suppressed.

    The first version of this function thresholded the raw Sato response at a global percentile,
    and recall against the human polygons came out at 0.000. The reason is visible in the
    diagnostic figure below: the strongest ridges in a tight-cropped candled egg are the dark
    silhouette boundary and the fill seam, not the crack, so the top percentile was entirely
    consumed by the egg outline and nothing was left for the crack itself.

    The fix runs Sato twice. Bright ridges at small scales are crack candidates. Dark ridges at
    larger scales are the silhouette and other broad dark structure; those are dilated and
    subtracted. A border band is also zeroed.
    """
    gf = g.astype(np.float32) / 255.0
    bright = sato(gf, sigmas=RIDGE_SIGMAS, black_ridges=False)
    dark = sato(gf, sigmas=DARK_SIGMAS, black_ridges=True)

    silhouette = (dark > np.percentile(dark, SILHOUETTE_PCTL)).astype(np.uint8)
    k = max(3, int(0.02 * g.shape[0]) | 1)
    silhouette = cv2.dilate(silhouette, np.ones((k, k), np.uint8))
    bright = bright.copy()
    bright[silhouette.astype(bool)] = 0.0

    b = max(2, int(BORDER_FRAC * g.shape[0]))
    bright[:b, :] = 0; bright[-b:, :] = 0; bright[:, :b] = 0; bright[:, -b:] = 0
    return bright


def ridge_features(rr):
    nz = rr[rr > 0]
    if nz.size < 10:
        nz = rr.ravel()
    p = np.percentile(nz, [50, 90, 99, 99.9])
    return {"ridge_mean": float(nz.mean()), "ridge_std": float(nz.std()),
            "ridge_p50": float(p[0]), "ridge_p90": float(p[1]),
            "ridge_p99": float(p[2]), "ridge_p999": float(p[3]),
            "ridge_frac_hi": float((rr > p[2]).mean()),
            "ridge_skew": float(((nz - nz.mean()) ** 3).mean() / (nz.std() ** 3 + 1e-9))}


def auto_segment(rr, pctl=None, min_px=MIN_COMPONENT_PX):
    """Threshold the silhouette-suppressed ridge response into a candidate crack mask."""
    pctl = RIDGE_PCTL if pctl is None else pctl
    valid = rr > 0
    if valid.sum() < 50:
        return np.zeros_like(rr, np.uint8)
    t = np.percentile(rr[valid], pctl)
    m = ((rr > t) & valid).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    m = remove_small_objects(m.astype(bool), min_size=min_px).astype(np.uint8)
    return m


def segment_geometry(m):
    """Geometry of the automatically segmented candidate cracks.

    Deliberately mirrors the polygon features of notebook 01 -- elongation, area, fragment count
    and spatial spread -- so that B3 and B4 are measuring the same construct and the gap between
    them is interpretable.
    """
    n, lab, st, cent = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return {"seg_n": 0, "seg_area_frac": 0.0, "seg_area_largest": 0.0,
                "seg_elong_max": 1.0, "seg_elong_mean": 1.0, "seg_compactness": 0.0,
                "seg_spread": 0.0, "seg_fill": 0.0}
    areas, elongs = [], []
    for k in range(1, n):
        comp = (lab == k).astype(np.uint8)
        pts = cv2.findNonZero(comp)
        (_, _), (w, h), _ = cv2.minAreaRect(pts)
        lo, hi = sorted([float(w), float(h)])
        areas.append(float(st[k, cv2.CC_STAT_AREA]))
        elongs.append(hi / lo if lo > 1e-6 else 1.0)
    areas = np.asarray(areas); elongs = np.asarray(elongs)
    cs = cent[1:]
    total = float(areas.sum()); px = m.size
    per = float(cv2.arcLength(
        cv2.findNonZero(m), True)) if m.any() else 1.0
    return {"seg_n": int(n - 1), "seg_area_frac": total / px,
            "seg_area_largest": float(areas.max()) / px,
            "seg_elong_max": float(elongs.max()), "seg_elong_mean": float(elongs.mean()),
            "seg_compactness": float(4 * np.pi * total / max(per ** 2, 1e-6)),
            "seg_spread": float(np.hypot(*cs.std(axis=0))) / m.shape[0] * 100 if len(cs) > 1 else 0.0,
            "seg_fill": float(areas.max() / max(total, 1e-6))}


def polygon_mask(crack_region, side=FEAT_SIDE):
    m = np.zeros((side, side), np.uint8)
    if isinstance(crack_region, str) and crack_region.strip():
        for o in json.loads(crack_region):
            p = o.get("points", [])
            if len(p) >= 3:
                pts = np.round(np.asarray(p, float) / 100.0 * side).astype(np.int32)
                cv2.fillPoly(m, [pts], 1)
    return m


def extract(row):
    g = prep_gray(int(row["cache_row"]))
    rr = ridge_response(g)
    seg = auto_segment(rr)
    feats = {}
    feats.update(texture_features(g))
    feats.update(ridge_features(rr))
    feats.update(segment_geometry(seg))
    # agreement between the automatic segmentation and the human annotation
    hm = polygon_mask(row["crack_region"])
    inter = float((seg & hm).sum()); union = float((seg | hm).sum())
    feats["seg_iou_vs_human"] = inter / union if union > 0 else np.nan
    feats["seg_recall_vs_human"] = inter / hm.sum() if hm.sum() > 0 else np.nan
    feats["seg_precision_vs_human"] = inter / seg.sum() if seg.sum() > 0 else np.nan
    return feats

# --- give the classical detector its best shot: sweep the threshold ------
sweep_rows = []
sample = dev[dev["crack_pattern"].notna()].sample(min(120, int(dev["crack_pattern"].notna().sum())),
                                                  random_state=CFG.RANDOM_SEED)
cached = [(prep_gray(int(r["cache_row"])), polygon_mask(r["crack_region"]))
          for _, r in sample.iterrows()]
cached = [(ridge_response(g), hm) for g, hm in cached]
for pctl in [95.0, 97.0, 98.0, 99.0, 99.5]:
    ious, recs = [], []
    for rr, hm in cached:
        seg = auto_segment(rr, pctl)
        inter = float((seg & hm).sum()); union = float((seg | hm).sum())
        ious.append(inter / union if union > 0 else np.nan)
        recs.append(inter / hm.sum() if hm.sum() > 0 else np.nan)
    sweep_rows.append({"percentile": pctl, "median_iou": float(np.nanmedian(ious)),
                       "median_recall": float(np.nanmedian(recs)),
                       "mean_recall": float(np.nanmean(recs))})
sweep = pd.DataFrame(sweep_rows).set_index("percentile").round(4)
savetable(sweep, "t00_ridge_threshold_sweep",
          "Automatic segmentation quality across candidate ridge thresholds")
print(sweep.to_string())
RIDGE_PCTL = float(sweep["median_iou"].idxmax())
print(f"\nadopted ridge threshold percentile: {RIDGE_PCTL}")
S["ridge_threshold"] = RIDGE_PCTL
S["ridge_sweep"] = sweep.reset_index().to_dict("records")

t0 = time.time()
F = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(extract)(r) for _, r in dev.iterrows()))
print(f"\nextracted {F.shape[1]} features for {F.shape[0]} eggs in {time.time() - t0:.0f}s")

TEX = [c for c in F.columns if c.startswith(("lbp", "gabor"))]
RIDGE = [c for c in F.columns if c.startswith("ridge_")]
SEG = [c for c in F.columns if c.startswith("seg_") and "vs_human" not in c]
print(f"texture {len(TEX)} | ridge {len(RIDGE)} | segmentation geometry {len(SEG)}")

---
## 2. How well does the automatic crack detector actually find cracks?

Before the geometry features are used, the segmentation that produced them is checked against the
human polygons. This number matters twice: it tells us whether B3 is a fair classical competitor,
and it gives notebook 09 a non-deep reference point for the Grad-CAM localisation scores.

The polygons are generous outlines drawn *around* the crack rather than tight traces of it, so
recall is the meaningful measure here and precision will look poor by construction. That caveat is
stated in the dissertation rather than left for a reader to infer.

In [ ]:
cr = dev["crack_pattern"].notna().to_numpy()
seg_q = F.loc[cr, ["seg_iou_vs_human", "seg_recall_vs_human", "seg_precision_vs_human"]]
seg_q = seg_q.assign(pattern=dev.loc[cr, "crack_pattern"].values)
qt = seg_q.groupby("pattern").agg(["median", "mean"]).round(3)
savetable(qt, "t01_auto_segmentation_quality",
          "Agreement between the automatic ridge segmentation and the human annotation polygons")
S["auto_segmentation"] = {
    "median_iou": float(seg_q["seg_iou_vs_human"].median()),
    "median_recall": float(seg_q["seg_recall_vs_human"].median()),
    "by_pattern": qt["seg_recall_vs_human"]["median"].to_dict(),
}
print(qt.to_string())
med_rec = float(seg_q["seg_recall_vs_human"].median())
print(f"\nmedian IoU vs human polygons    : {seg_q['seg_iou_vs_human'].median():.3f}")
print(f"median recall of annotated area : {med_rec:.3f}")
if med_rec < 0.10:
    print("\nThe classical detector does NOT find the cracks. Treat B3 accordingly: whatever it")
    print("scores comes from global texture statistics, not from crack morphology, and the")
    print("dissertation must say so rather than presenting B3 as a working crack pipeline.")
    print("This failure is itself a result -- it is the evidence that localising micro-cracks")
    print("under candling is the hard part, and the direct argument for a learned approach.")
elif med_rec < 0.35:
    print("\nPartial localisation: B3 is a weak but genuine crack pipeline.")
else:
    print("\nThe classical detector localises cracks competently; B3 is a serious competitor")
    print("and the deep model must beat it on morphology, not merely on detection.")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
ax = axes[0]
for p in CFG.PATTERNS:
    v = seg_q.loc[seg_q["pattern"].eq(p), "seg_recall_vs_human"].dropna()
    ax.hist(v, bins=22, alpha=0.6, color=CFG.PALETTE[p], label=p)
ax.set_xlabel("recall of annotated crack area"); ax.set_ylabel("eggs")
ax.set_title("(a) Automatic detector recall"); ax.legend(fontsize=8)

# a worked example per pattern, with the ridge response shown so failures are diagnosable
for ax, p in zip(axes[1:], CFG.PATTERNS):
    sub = dev[dev["crack_pattern"].eq(p)]
    row = sub.iloc[len(sub) // 2]
    g = prep_gray(int(row["cache_row"]))
    rr = ridge_response(g); seg = auto_segment(rr); hm = polygon_mask(row["crack_region"])
    base = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB).astype(np.float32)
    heat = cv2.applyColorMap(
        np.clip(rr / (np.percentile(rr[rr > 0], 99.5) + 1e-9) * 255, 0, 255).astype(np.uint8),
        cv2.COLORMAP_INFERNO)[:, :, ::-1].astype(np.float32)
    rgb = (0.55 * base + 0.45 * heat).astype(np.uint8)
    rgb[seg.astype(bool)] = [0, 229, 255]
    cont, _ = cv2.findContours(hm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(rgb, cont, -1, (255, 220, 0), 2)
    ax.imshow(rgb); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    rec = float((seg & hm).sum()) / max(float(hm.sum()), 1)
    ax.set_title(f"{p} — {row['egg_id']}  recall {rec:.2f}\n"
                 f"heat: ridge response, cyan: detected, yellow: human", fontsize=8)
fig.suptitle("Does the classical ridge detector find the crack?", y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "auto_segmentation", "Automatic ridge segmentation against the human annotations")
plt.show()

---
## 3. The baselines

Every baseline is evaluated on the fold assignment written by notebook 02, so results are directly
comparable with the models in notebook 05. Each is reported as mean ± standard deviation across the
five folds, because a single number from a single split is not a result.

Both split regimes are run: the stratified folds (primary) and leave-capture-block-out (secondary).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut

post_std = pd.read_csv(PROC / "post_stats_standardised.csv")
post_std = post_std.set_index("egg_id").loc[dev["egg_id"]].reset_index()
CAPTURE = [c for c in post_std.columns if c != "egg_id"]

POLY = [c for c in ["poly_n", "poly_area_pct", "poly_elongation_max", "poly_spread"]
        if c in dev.columns]

FEATURE_SETS = {
    "B1 capture statistics only": post_std[CAPTURE].to_numpy(float),
    "B2 texture (LBP + Gabor)": F[TEX].to_numpy(float),
    "B2b texture + ridge response": F[TEX + RIDGE].to_numpy(float),
    "B3 automatic crack geometry": F[RIDGE + SEG].to_numpy(float),
    "B3b texture + ridge + geometry": F[TEX + RIDGE + SEG].to_numpy(float),
    "B4 human polygon geometry (oracle)": dev[POLY].fillna(0.0).to_numpy(float),
}

CLASSIFIERS = {
    "logreg": lambda: make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(max_iter=4000, class_weight="balanced", C=0.5,
                           random_state=CFG.RANDOM_SEED)),
    "linsvc": lambda: make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        LinearSVC(class_weight="balanced", C=0.1, max_iter=8000,
                  random_state=CFG.RANDOM_SEED)),
    "histgb": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                       random_state=CFG.RANDOM_SEED)),
}

def cv_scores(X, y, make_clf, fold_ids):
    """Macro F1 and balanced accuracy per fold, using the predefined assignment."""
    f1s, bas = [], []
    for k in sorted(set(fold_ids)):
        tr, va = fold_ids != k, fold_ids == k
        if len(set(y[tr])) < 2 or va.sum() == 0:
            continue
        clf = make_clf().fit(X[tr], y[tr])
        p = clf.predict(X[va])
        f1s.append(f1_score(y[va], p, average="macro"))
        bas.append(balanced_accuracy_score(y[va], p))
    return np.array(f1s), np.array(bas)

rows = []
for task, y in [("4-class", y4), ("binary", y2)]:
    # B0
    for strat in ["most_frequent", "stratified"]:
        f1s, bas = cv_scores(np.zeros((len(dev), 1)), y,
                             lambda s=strat: DummyClassifier(strategy=s,
                                                             random_state=CFG.RANDOM_SEED), folds)
        rows.append({"task": task, "baseline": f"B0 {strat}", "classifier": "-",
                     "macro_f1_mean": f1s.mean(), "macro_f1_sd": f1s.std(),
                     "bal_acc_mean": bas.mean()})
    for name, X in FEATURE_SETS.items():
        # The oracle is degenerate on the binary task: a polygon exists if and only if the egg is
        # cracked, so B4 would score a meaningless 1.000. It is informative only for morphology.
        if name.startswith("B4") and task == "binary":
            rows.append({"task": task, "baseline": name, "classifier": "-",
                         "macro_f1_mean": np.nan, "macro_f1_sd": np.nan, "bal_acc_mean": np.nan})
            continue
        for cname, mk in CLASSIFIERS.items():
            f1s, bas = cv_scores(X, y, mk, folds)
            rows.append({"task": task, "baseline": name, "classifier": cname,
                         "macro_f1_mean": f1s.mean(), "macro_f1_sd": f1s.std(),
                         "bal_acc_mean": bas.mean()})
    print(f"  {task} done")

res = pd.DataFrame(rows)
res[["macro_f1_mean", "macro_f1_sd", "bal_acc_mean"]] = \
    res[["macro_f1_mean", "macro_f1_sd", "bal_acc_mean"]].round(3)
savetable(res.set_index(["task", "baseline", "classifier"]), "t02_all_baselines",
          "All baselines under the notebook 02 fold assignment")

best = (res.sort_values("macro_f1_mean", ascending=False)
          .groupby(["task", "baseline"], as_index=False).first())
piv = best.pivot(index="baseline", columns="task", values="macro_f1_mean")
sd = best.pivot(index="baseline", columns="task", values="macro_f1_sd")
clf = best.pivot(index="baseline", columns="task", values="classifier")
def _fmt(b, t):
    m, e = piv.loc[b, t], sd.loc[b, t]
    return "n/a (degenerate)" if pd.isna(m) else f"{m:.3f} ± {e:.3f}"

summary = pd.DataFrame({
    "4-class macro F1": [_fmt(b, "4-class") for b in piv.index],
    "4-class clf": [clf.loc[b, "4-class"] for b in piv.index],
    "binary macro F1": [_fmt(b, "binary") for b in piv.index],
    "binary clf": [clf.loc[b, "binary"] for b in piv.index],
}, index=piv.index)
savetable(summary, "t03_baseline_summary", "Best classifier per baseline family")
print()
print(summary.to_string())
S["baselines"] = {b: {"four_class": float(piv.loc[b, "4-class"]),
                      "binary": float(piv.loc[b, "binary"])} for b in piv.index}

In [ ]:
# --- secondary regime: leave one capture block out ------------------------
logo_rows = []
for task, y in [("4-class", y4), ("binary", y2)]:
    for name, X in FEATURE_SETS.items():
        cname = str(best.loc[(best["task"].eq(task)) & (best["baseline"].eq(name)),
                             "classifier"].iloc[0])
        if cname not in CLASSIFIERS:
            logo_rows.append({"task": task, "baseline": name,
                              "macro_f1_mean": np.nan, "macro_f1_sd": np.nan})
            continue
        mk = CLASSIFIERS[cname]
        f1s, _ = cv_scores(X, y, mk, blocks)
        logo_rows.append({"task": task, "baseline": name,
                          "macro_f1_mean": round(float(f1s.mean()), 3),
                          "macro_f1_sd": round(float(f1s.std()), 3)})
logo = pd.DataFrame(logo_rows).pivot(index="baseline", columns="task", values="macro_f1_mean")
logo.columns = [f"{c} (session-grouped)" for c in logo.columns]
comp = pd.concat([piv.rename(columns=lambda c: f"{c} (stratified)"), logo], axis=1).round(3)
savetable(comp, "t04_baselines_both_regimes",
          "Baseline performance under both split regimes")
S["baselines_grouped"] = logo.round(3).to_dict()
print(comp.to_string())

In [ ]:
order = [b for b in ["B0 most_frequent", "B1 capture statistics only", "B2 texture (LBP + Gabor)",
                     "B2b texture + ridge response", "B3 automatic crack geometry",
                     "B3b texture + ridge + geometry", "B4 human polygon geometry (oracle)"]
         if b in set(best["baseline"])]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for ax, task in zip(axes, ["4-class", "binary"]):
    b = best[best["task"].eq(task)].set_index("baseline").reindex(order)
    colors = ["#BBBBBB" if "B0" in i else "#C44E52" if "B1" in i
              else "#937860" if "B4" in i else "#4C72B0" for i in b.index]
    ax.barh(range(len(b)), b["macro_f1_mean"].fillna(0), xerr=b["macro_f1_sd"].fillna(0),
            color=colors, alpha=0.9, capsize=3)
    floor = b.loc["B1 capture statistics only", "macro_f1_mean"] if \
        "B1 capture statistics only" in b.index else np.nan
    ax.axvline(floor, ls="--", color="#C44E52", lw=1.4)
    ax.text(floor + 0.008, -0.6, "acquisition floor", fontsize=8, color="#C44E52")
    for i, (m, e) in enumerate(zip(b["macro_f1_mean"], b["macro_f1_sd"])):
        ax.text(0.012 if pd.isna(m) else m + e + 0.012, i,
                "n/a (degenerate)" if pd.isna(m) else f"{m:.3f}", va="center", fontsize=8)
    ax.set_yticks(range(len(b)))
    ax.set_yticklabels([i.replace(" (", "\n(") for i in b.index], fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel("macro F1 (mean ± SD over 5 folds)")
    ax.set_title(f"{task}"); ax.set_xlim(0, 1.0)
fig.suptitle("Baselines — everything must be read against the acquisition floor, not against chance",
             y=1.02, fontsize=12)
fig.tight_layout()
savefig(fig, "baseline_comparison",
        "Baseline macro F1 for both tasks, with the capture-statistics floor marked")
plt.show()

---
## 4. The target notebook 05 must beat

In [ ]:
floor4 = S["baselines"].get("B1 capture statistics only", {}).get("four_class", np.nan)
floor2 = S["baselines"].get("B1 capture statistics only", {}).get("binary", np.nan)
best_classical4 = max(v["four_class"] for k, v in S["baselines"].items()
                      if k.startswith(("B2", "B3")))
best_classical2 = max(v["binary"] for k, v in S["baselines"].items()
                      if k.startswith(("B2", "B3")))
oracle4 = S["baselines"].get("B4 human polygon geometry (oracle)", {}).get("four_class", np.nan)

S["targets"] = {"acquisition_floor_4class": floor4, "acquisition_floor_binary": floor2,
                "best_classical_4class": best_classical4, "best_classical_binary": best_classical2,
                "oracle_4class": oracle4}

print("TARGETS FOR NOTEBOOK 05")
print("=" * 58)
print(f"  acquisition floor  4-class {floor4:.3f}   binary {floor2:.3f}")
print(f"  best classical     4-class {best_classical4:.3f}   binary {best_classical2:.3f}")
print(f"  oracle ceiling     4-class {oracle4:.3f}   (human polygons; not deployable)")
print()
print("A deep model must clear the best classical baseline by a margin larger than the fold")
print("standard deviation to count as an improvement. Clearing only the acquisition floor means")
print("it has learned the camera, not the crack.")
print()
gap = oracle4 - best_classical4
print(f"Gap between the oracle and the best deployable classical pipeline: {gap:.3f}")
if gap > 0.15:
    print("That gap is large, which means most of the difficulty is in LOCATING the crack rather")
    print("than in classifying its shape once found. This is the argument for formulation (d) in")
    print("notebook 05 -- supervising the backbone with the annotation polygons -- and it should")
    print("be stated in Section 4.6.1 as the reason that formulation was included.")
else:
    print("That gap is small, so locating the crack is not the bottleneck; the morphology")
    print("distinction itself is. Notebook 05 should prioritise class-imbalance handling and")
    print("fine-grained feature resolution over segmentation supervision.")

In [ ]:
S["figures"], S["tables"] = FIGS, TABS
def _safe(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)
(OUT / "baseline_summary.json").write_text(json.dumps(S, indent=2, default=_safe))
F.assign(egg_id=dev["egg_id"].values).to_csv(PROC / "classical_features.csv", index=False)

L = ["=" * 70, "NOTEBOOK 04 DIGEST -- BASELINES", "=" * 70,
     f"generated: {S['generated_at']}", "",
     "AUTOMATIC CRACK SEGMENTATION vs HUMAN POLYGONS",
     f"   median IoU    {S['auto_segmentation']['median_iou']:.3f}",
     f"   median recall {S['auto_segmentation']['median_recall']:.3f}",
     f"   recall by pattern {S['auto_segmentation']['by_pattern']}", "",
     "BASELINE MACRO F1 (mean over 5 folds, stratified regime)"]
for b, v in S["baselines"].items():
    L.append(f"   {b:<40s} 4cls={v['four_class']:.3f}  bin={v['binary']:.3f}")
L += ["", "TARGETS FOR NOTEBOOK 05"]
for k, v in S["targets"].items():
    L.append(f"   {k:<30s} {v:.3f}")
L.append("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print(digest)